# Tutorial 5 — Convolutional Networks for Geometric Data

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 5: Convolutional Neural Networks**

---

Tutorial 4 made the hypothesis space bigger and lost every guarantee. Lecture 5
does the opposite: it makes the space **smaller**, by insisting that the linear maps
commute with a group action.

A convolution is precisely a linear map $\mathbb{R}^{\Lambda} \to \mathbb{R}^{\Lambda}$
that commutes with translation. That single constraint buys weight sharing, locality,
and a parameter count independent of image size — and it is the first instance in
this course of the principle that Lecture 10 will state in general: *build the
symmetry into the architecture rather than hoping the model learns it*.

We test that claim on geometric raster data: images of plane curves, labelled by
quantities we can compute exactly.

| § | Question |
|---|---|
| 1 | In what sense is a convolution the translation-equivariant map? |
| 2 | A dataset: curves rasterised into images, with exact labels |
| 3 | CNN vs MLP — accuracy, parameters, and behaviour under translation |
| 4 | What the filters learn |
| 5 | Summary |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 20260905
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("torch", torch.__version__)

---
## 1. Convolution is the translation-equivariant linear map

Let $T_v$ denote translation of an image by $v$. A linear map $K$ is
**equivariant** if it commutes with the action,

$$K \circ T_v \;=\; T_v \circ K \qquad \text{for all } v .$$

The theorem behind Lecture 5 is that on a group (here $\mathbb{Z}^2$, or
$(\mathbb{Z}/n)^2$ with periodic boundaries) *every* such map is a convolution.
Equivalently: convolutions are exactly the operators diagonalised by the Fourier
transform, which is the character theory of the group.

The practical consequences follow from the constraint, not from any empirical
observation:

- **Weight sharing.** A general linear map on an $n\times n$ image needs $n^4$
  parameters; a $k\times k$ convolution needs $k^2$, whatever $n$ is.
- **Locality** is a *separate* assumption — restricting the kernel's support. It is
  what makes convolutions cheap, and what makes them appropriate for data where
  interactions are local.
- **Equivariance, not invariance.** A convolution moves features around with the
  image. Invariance comes later, from pooling or a global reduction.

Worth verifying rather than believing.

In [ ]:
def roll2d(img, dy, dx):
    """Translate an image cyclically — the action of (Z/n)^2."""
    return torch.roll(img, shifts=(dy, dx), dims=(-2, -1))


conv = nn.Conv2d(1, 3, kernel_size=5, padding=2, padding_mode="circular", bias=False).float()
pool = nn.AdaptiveAvgPool2d(1)

x = torch.tensor(rng.normal(size=(1, 1, 32, 32)), dtype=torch.float32)
dy, dx = 7, -3

lhs = conv(roll2d(x, dy, dx))          # convolve the translated image
rhs = roll2d(conv(x), dy, dx)          # translate the convolved image
print(f"equivariance   ||K(T x) - T(K x)||_inf = {(lhs - rhs).abs().max():.2e}")

print(f"invariance of a global mean pool: "
      f"{(pool(conv(roll2d(x, dy, dx))) - pool(conv(x))).abs().max():.2e}")

dense = nn.Linear(32 * 32, 3 * 32 * 32, bias=False).float()
l2 = dense(roll2d(x, dy, dx).reshape(1, -1))
r2 = roll2d(dense(x.reshape(1, -1)).reshape(1, 3, 32, 32), dy, dx).reshape(1, -1)
print(f"\na dense layer of the same shape: ||.||_inf = {(l2 - r2).abs().max():.2e}"
      f"   ({dense.weight.numel():,} parameters vs {conv.weight.numel()} for the conv)")

Equivariance holds to machine precision for the convolution and fails completely for
a dense layer with the same input and output shape — which uses about $10{,}000$
times as many parameters to be worse. The global average pool converts equivariance
into exact **invariance**, which is the standard way to end a CNN when the label
does not depend on position.

---
## 2. A dataset of rasterised curves

We reuse Tutorial 3's random plane curves — polar Fourier series
$r(\theta) = 1 + \sum_k (a_k\cos k\theta + b_k \sin k\theta)$ — but present them to
the model as **images** rather than coefficients. Each curve is drawn into an
$n\times n$ raster as a filled indicator of its interior.

Two labels, both exact:

- **area**, $A = \pi + \frac{\pi}{2}\sum_k (a_k^2+b_k^2)$ (regression);
- **convex or not**, from the sign of $r^2 + 2r'^2 - rr''$ (classification).

Presenting geometry as pixels is a real modelling choice with real costs — we are
discarding exact coefficients in favour of a lossy raster. The point of the exercise
is that this is the situation you are in whenever the data arrives as a picture: a
microscope scan, a rendered surface, a discretised field.

In [ ]:
GRID = 32
K_MODES, DECAY, SIGMA = 8, 2.5, 0.25
KS = np.arange(1, K_MODES + 1)
M_TH = 512
TH = np.linspace(0, 2 * np.pi, M_TH, endpoint=False)
_C, _S = np.cos(np.outer(KS, TH)), np.sin(np.outer(KS, TH))

gy, gx = np.mgrid[0:GRID, 0:GRID]
px = (gx + 0.5) / GRID * 3.0 - 1.5           # the image covers [-1.5, 1.5]^2
py = (gy + 0.5) / GRID * 3.0 - 1.5
p_rad = np.sqrt(px**2 + py**2)
p_ang = np.arctan2(py, px) % (2 * np.pi)
_bin = np.clip((p_ang / (2 * np.pi) * M_TH).astype(int), 0, M_TH - 1)


def make_dataset(n, rng, shift=False):
    """Rasterised random curves, with exact area and convexity labels."""
    scale = SIGMA / KS**DECAY
    a = rng.normal(size=(n, K_MODES)) * scale
    b = rng.normal(size=(n, K_MODES)) * scale
    r = 1 + a @ _C + b @ _S
    r1 = (-a * KS) @ _S + (b * KS) @ _C
    r2 = (-a * KS**2) @ _C + (-b * KS**2) @ _S

    imgs = (p_rad[None, :, :] <= r[:, _bin]).astype(np.float32)
    if shift:                                 # random cyclic translation
        for i in range(n):
            imgs[i] = np.roll(imgs[i], (rng.integers(-8, 9), rng.integers(-8, 9)), (0, 1))
    area = np.pi + 0.5 * np.pi * np.sum(a**2 + b**2, axis=1)
    convex = ((r**2 + 2 * r1**2 - r * r2).min(axis=1) >= 0).astype(np.float32)
    return imgs[:, None, :, :], area, convex


Xtr, Atr, Ctr = make_dataset(2500, rng)
Xte, Ate, Cte = make_dataset(1500, rng)
Xte_sh, Ate_sh, Cte_sh = make_dataset(1500, np.random.default_rng(99), shift=True)

print(f"images {Xtr.shape},  convex fraction {Ctr.mean():.1%}")
print(f"area range [{Atr.min():.2f}, {Atr.max():.2f}]")
print(f"pixel-count vs true area, correlation {np.corrcoef(Xtr.sum((1,2,3)), Atr)[0,1]:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10.5, 3.7))
for j, ax in enumerate(axes.ravel()):
    src = (Xtr, Ctr, Atr) if j < 6 else (Xte_sh, Cte_sh, Ate_sh)
    k = j if j < 6 else j - 6
    ax.imshow(src[0][k, 0], cmap="Greys", origin="lower")
    ax.set_title(f"{'convex' if src[1][k] > 0.5 else 'non-convex'}\nA={src[2][k]:.2f}",
                 fontsize=6.5, color=GEO_TEAL if src[1][k] > 0.5 else GEO_RUST)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
axes[0, 0].set_ylabel("centred", fontsize=8)
axes[1, 0].set_ylabel("translated", fontsize=8)
fig.suptitle("rasterised curves: training set (top), shifted test set (bottom)", y=1.02)
plt.tight_layout(); plt.show()

---
## 3. CNN versus MLP

Now the comparison. Both models see identical images and identical labels; only
the architecture differs. The CNN is built from $3\times3$ convolutions with
circular padding (matching the cyclic translation action of §1) and ends in a global
average pool, so it is invariant to translation by construction. The MLP flattens
the image and has no idea the pixels form a grid.

We test on two sets: one drawn like the training data, and one where every image has
been **randomly translated**. The labels are unchanged by translation, so a model
that has understood the geometry should not care.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, c=8):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, c, 3, padding=1, padding_mode="circular"), nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1, padding_mode="circular"), nn.ReLU(),
            nn.AvgPool2d(2),
            nn.Conv2d(c, 2 * c, 3, padding=1, padding_mode="circular"), nn.ReLU(),
        )
        self.head = nn.Linear(2 * c, 1)

    def forward(self, x):
        h = self.features(x)
        return self.head(h.mean(dim=(2, 3)))       # global pool -> invariance


def make_flat_mlp(width=64):
    return nn.Sequential(nn.Flatten(), nn.Linear(GRID * GRID, width), nn.ReLU(),
                         nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 1))


def train(model, X, y, epochs=25, bs=256, lr=3e-3, loss="mse"):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss() if loss == "mse" else nn.BCEWithLogitsLoss()
    n = len(Xt)
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            opt.zero_grad(); lossf(model(Xt[idx]), yt[idx]).backward(); opt.step()
    return model


def evaluate(model, X, y, kind):
    with torch.no_grad():
        out = model(torch.tensor(X, dtype=torch.float32)).numpy().ravel()
    if kind == "mse":
        return np.mean((out - y)**2)
    return np.mean((out > 0) == (y > 0.5))


nparams = lambda m: sum(p.numel() for p in m.parameters())

In [ ]:
torch.manual_seed(0); cnn_c = train(SmallCNN(), Xtr, Ctr, loss="bce")
torch.manual_seed(0); mlp_c = train(make_flat_mlp(), Xtr, Ctr, loss="bce")

rows = []
for name, m in [("CNN", cnn_c), ("MLP", mlp_c)]:
    rows.append((name, nparams(m),
                 evaluate(m, Xte, Cte, "acc"), evaluate(m, Xte_sh, Cte_sh, "acc")))

print("CONVEXITY (majority baseline "
      f"{max(Cte.mean(), 1-Cte.mean()):.1%})")
print(f"  {'model':6s} {'params':>8s} {'test acc':>10s} {'translated':>12s}")
for name, p, a, b in rows:
    print(f"  {name:6s} {p:8,d} {a:10.1%} {b:12.1%}")

In [ ]:
torch.manual_seed(0); cnn_a = train(SmallCNN(), Xtr, Atr, loss="mse")
torch.manual_seed(0); mlp_a = train(make_flat_mlp(), Xtr, Atr, loss="mse")
base = np.mean((Ate - Atr.mean())**2)

print(f"AREA  (predict-the-mean baseline MSE {base:.4f})")
print(f"  {'model':6s} {'params':>8s} {'test MSE':>10s} {'translated':>12s}")
for name, m in [("CNN", cnn_a), ("MLP", mlp_a)]:
    print(f"  {name:6s} {nparams(m):8,d} {evaluate(m, Xte, Ate, 'mse'):10.5f}"
          f" {evaluate(m, Xte_sh, Ate_sh, 'mse'):12.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.3))
labels = ["centred", "translated"]
w = 0.35
for k, (name, vals, col) in enumerate([("CNN", [rows[0][2], rows[0][3]], GEO_TEAL),
                                       ("MLP", [rows[1][2], rows[1][3]], GEO_RUST)]):
    axes[0].bar(np.arange(2) + (k - 0.5) * w, vals, w, label=name, color=col)
axes[0].axhline(max(Cte.mean(), 1 - Cte.mean()), color="0.4", ls="--", lw=1.2,
                label="majority baseline")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("accuracy"); axes[0].set_title("convexity"); axes[0].legend(fontsize=7.5)
axes[0].set_ylim(0, 1)

mses = [[evaluate(cnn_a, Xte, Ate, "mse"), evaluate(cnn_a, Xte_sh, Ate_sh, "mse")],
        [evaluate(mlp_a, Xte, Ate, "mse"), evaluate(mlp_a, Xte_sh, Ate_sh, "mse")]]
for k, (name, col) in enumerate([("CNN", GEO_TEAL), ("MLP", GEO_RUST)]):
    axes[1].bar(np.arange(2) + (k - 0.5) * w, mses[k], w, label=name, color=col)
axes[1].axhline(base, color="0.4", ls="--", lw=1.2, label="predict the mean")
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(labels)
axes[1].set_yscale("log"); axes[1].set_ylabel("test MSE")
axes[1].set_title("area"); axes[1].legend(fontsize=7.5)
plt.tight_layout(); plt.show()

On **centred** data the MLP is the better model, and by a clear margin — it has
almost forty times the parameters and it uses them. So this is not a story about
convolutions being uniformly superior.

On **translated** data the CNN is unchanged, to three significant figures, because
its output is invariant by construction: nothing about `Conv → ReLU → global mean`
can depend on where the shape sits. The MLP collapses — on area it does *worse than
predicting the mean*.

Now the observation worth pausing on. The area of these shapes is essentially the
number of filled pixels: we measured that correlation as $0.974$ in §2. Counting
pixels is a **translation-invariant** function, it is **linear**, and it therefore
sits comfortably inside the MLP's hypothesis space — a single layer of ones would
compute it. The MLP had every opportunity to find the invariant solution, and it did
not. It found a position-dependent shortcut that was slightly better on the training
distribution and worthless off it.

That is the real content of the geometric-deep-learning thesis, and it is stronger
than "constraints save parameters". Empirical risk minimisation optimises the
average over the training distribution, and nothing in that objective rewards a
solution for being invariant. If you want the symmetry, you generally have to impose
it — through the architecture, as here, or through augmentation, which spends
samples and capacity re-learning something you already knew. Lecture 10 generalises
this from translations to arbitrary group actions.

> **Exercise 1 — how much is the symmetry worth?**
> (a) Retrain the MLP with random translations applied during training
> (augmentation). How much of the gap does it close, and at what cost in epochs?
>
> (b) Sweep the training-set size for both models and plot accuracy against $N$.
> The claim in Lecture 5 is that the constrained model is more **sample-efficient**;
> confirm or refute it.
>
> (c) Replace `padding_mode="circular"` with the default zero padding. The
> equivariance of §1 is then only approximate near the boundary — measure how much
> accuracy on the translated set degrades, and explain why.

---
## 4. What the filters learn

Lecture 5 described convolution kernels as local feature detectors. It is tempting
to expect edge detectors, and textbook figures usually show them — but our images
are *indicator functions*, and for those there are two informative local statistics,
not one:

- **occupancy**: how much of this neighbourhood is inside the shape? A kernel with a
  large positive sum measures this, and integrating it gives the area.
- **boundary**: is there a transition here, and in which direction? A kernel whose
  entries sum to (near) zero annihilates constants and responds only at edges — a
  discrete difference operator.

Rather than assert which we get, let us measure. For a $3\times3$ kernel $k$,
compare $\lvert\sum_{ij} k_{ij}\rvert$ against the typical entry size
$9\,\overline{\lvert k\rvert}$: near zero means a difference operator, order one
means an occupancy detector.

In [ ]:
W = cnn_c.features[0].weight.detach().numpy()[:, 0]      # (channels, 3, 3)

fig = plt.figure(figsize=(11.0, 3.4))
for j in range(len(W)):
    ax = fig.add_subplot(2, 8, j + 1)
    m = np.abs(W[j]).max()
    ax.imshow(W[j], cmap="RdBu_r", vmin=-m, vmax=m)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(f"filter {j}", fontsize=6.5)

img = Xte[3, 0]
with torch.no_grad():
    act = cnn_c.features[0](torch.tensor(Xte[3:4], dtype=torch.float32)).numpy()[0]
for j in range(len(W)):
    ax = fig.add_subplot(2, 8, 8 + j + 1)
    ax.imshow(act[j], cmap="magma", origin="lower")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle("first-layer kernels (top) and their responses to one curve (bottom)", y=1.02)
plt.tight_layout(); plt.show()

sums = W.reshape(len(W), -1).sum(1)
scale = 9 * np.abs(W).reshape(len(W), -1).mean(1)      # typical size of a 3x3 sum
ratio = np.abs(sums) / scale

print(f"  {'filter':>7s} {'sum':>8s} {'|sum| / 9|k|':>14s}   character")
for j, (sm, rt) in enumerate(zip(sums, ratio)):
    kind = "difference (edge)" if rt < 0.35 else ("mixed" if rt < 0.7 else "occupancy (area)")
    print(f"  {j:7d} {sm:+8.3f} {rt:14.2f}   {kind}")
print(f"\n{int((ratio < 0.35).sum())} of {len(W)} kernels are difference operators; "
      f"{int((ratio >= 0.7).sum())} are occupancy detectors")

The filters come out **mixed**, and mostly on the occupancy side — which is the
honest answer, and not the one the textbook picture would have predicted. The
response maps show why: for an indicator image, a positive-sum kernel lights up the
whole interior, and averaging that over the image is very nearly the area, which is
exactly what one of our two labels is.

Two lessons, and the second is the more useful one.

First, the network learns the features its **loss** asks for, not the ones we find
photogenic. Train on a label that depends on the boundary and edge-like kernels are
rewarded; train on one that depends on bulk and occupancy kernels are. Exercise 2(b)
makes the comparison directly.

Second, and more generally: **inspect what your model learned rather than assuming
it matches the picture in the lecture.** The statistic above took three lines. A
claim like "the first layer learns edge detectors" is checkable, and in this
particular setting it is largely false — which is worth more to you than a
confirmation would have been.

Where zero-sum kernels *do* appear, they are finite-difference stencils: the network
has discretised a first-order differential operator. Lecture 11 returns to this from
the other direction, building networks out of discretised operators rather than
hoping to recover them.

> **Exercise 2 — read the filters as operators.**
> (a) Compute each kernel's sum and its first moments $\sum_{ij} k_{ij}\,i$ and
> $\sum_{ij} k_{ij}\,j$. Which channels approximate $\partial_x$, which $\partial_y$,
> and are there any that approximate a Laplacian (zero sum *and* zero first moments)?
>
> (b) Train on the *area* target instead and compare the filters. Does a different
> label produce different features, or does the boundary dominate either way?
>
> (c) Feed the network a curve and its reflection. The architecture is equivariant
> to translation but **not** to rotation or reflection — measure how much the
> prediction changes, and propose an architectural fix.

---
## 5. What to take away

- A convolution is **the** translation-equivariant linear map, not merely a
  convenient one. Everything else — weight sharing, size-independent parameter
  counts — follows from that constraint.
- **Equivariance is not invariance.** Features move with the image; invariance
  arrives only when you reduce over position, here with a global average pool.
- **Constraints beat capacity when the constraint is true.** The CNN and MLP were
  comparable on centred data and utterly different on translated data, at a
  thousandth of the parameter count.
- **Augmentation is the expensive alternative** to a symmetry you already know: it
  spends samples and capacity re-learning something the architecture could have had
  for free.
- Learned kernels turn out to be **difference operators** — the network discretises
  a derivative because the geometry of the data leaves nothing else to detect.

### Next

**Lecture 6** replaces the fixed neighbourhood of a convolution with a learned,
data-dependent one: attention. **Tutorial 6** applies it to structured geometric
data where there is no grid at all.

### Further reading

- Cohen & Welling, "Group equivariant convolutional networks", *ICML* 2016 — the generalisation §1 is a special case of.
- Bronstein, Bruna, Cohen & Veličković, *Geometric Deep Learning* (2021), ch. 3–5.
- Zhang, "Making convolutional networks shift-invariant again", *ICML* 2019 — why real CNNs are less equivariant than the theory suggests, because of striding.